In [ ]:
import time
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from llm5w1h import NewsArticle, NewsAnalyzer

model = "llama3.1"

def extract_texts(file_path, data_list):
    # extracts specified component from json object
    def extract_true_component(data, component):
        annotations = data["fiveWoneH"][component]["annotated"]
        texts = [item.get("text") for item in annotations]
        return "; ".join(text for text in texts if text is not None)

    # loads json object from file
    with open(file_path, "r") as file:
        data = json.load(file)

    # extracts true components from the annotated articles (in a json object)
    text = data["text"]
    what_true = extract_true_component(data, "what")
    where_true = extract_true_component(data, "where")
    when_true = extract_true_component(data, "when")
    who_true = extract_true_component(data, "who")
    why_true = extract_true_component(data, "why")
    how_true = extract_true_component(data, "how")

    analyzer = NewsAnalyzer("http://localhost:11434","api_key", model)
    article = NewsArticle(data.get("title"), data.get("description"), text, data.get("date_publish"), data.get("url"))
    analyzer.process_article(article)

    extracted_components = analyzer.extract_components()

    data_list.append({
        "text": text,
        "what_true": what_true,
        "where_true": where_true,
        "when_true": when_true,
        "who_true": who_true,
        "why_true": why_true,
        "how_true": how_true,
        "what_pred": extracted_components["what_pred"],
        "where_pred": extracted_components["where_pred"],
        "when_pred": extracted_components["when_pred"],
        "who_pred": extracted_components["who_pred"],
        "why_pred": extracted_components["why_pred"],
        "how_pred": extracted_components["how_pred"]
    })

In [ ]:
# testing with a single sample
data_list = []
extract_texts("./data_samples/0e5fa7c0e6252bfeeea5e3840c6cb503f299c19d24331c4ba60c5974.json", data_list)
print(data_list)

In [ ]:
pd.DataFrame(data_list)

In [ ]:
from bert_score import score

def evaluate(data_list):
    e = []
    global_min = 1
    global_max = 0
    for article in data_list:
        cands = [article[component] for component in article if component.endswith("_pred")]
        refs = [article[component] for component in article if component.endswith("_true")]
        P, R, F1 = score(cands, refs, lang="en")

        nonzero_F1 = F1[F1 > 0] # filters 0 values, which refer to components with no '_true' corresponding component

        # updates global min and global max values
        if nonzero_F1.min() < global_min:
            global_min = nonzero_F1.min()
        if nonzero_F1.max() > global_max:
            global_max = nonzero_F1.max()

        print(F1)
        mean = nonzero_F1.mean()
        print(f"System level F1 score: {mean:.3f}")
        e.append([article, cands, refs, P, R, F1, mean])

    global_avg = np.mean([article[-1] for article in e])
    print(f"Global minimum F1 score: {global_min:.3f}")
    print(f"Global maximum F1 score: {global_max:.3f}")
    print(f"Arithmetic average of mean F1 scores: {global_avg:.3f}")
    return e

In [ ]:
evaluate(data_list)

In [ ]:
# extracting components for all articles and writing results in a spreadsheet
data_list = []
data_folder = './data_samples/'
for filename in tqdm(os.listdir(data_folder)):
    file_path = os.path.join(data_folder, filename)
    extract_texts(file_path, data_list)

In [ ]:
df = pd.DataFrame(data_list)
df.to_excel('news_5w1h.xlsx', index=False)
df.to_csv('news_5w1h.csv', index=False, encoding='utf-8')

In [ ]:
# gets articles and their extracted components from the spreadsheets and compares extracted components with the true (annotated) components
df = pd.read_excel("news_5w1h.xlsx")
df = df.fillna("")
data_list = df.to_dict(orient="records")
e = evaluate(data_list)
df_e = pd.DataFrame(e)
df_e['model'] = 'llama3.1'

In [ ]:
df_e.to_pickle("news_5w1h_avaliacao.pkl")
df_e.to_excel('news_5w1h_avaliacao.xlsx')